# Fine-tune Qwen2.5-0.5B-Instruct for support ticket triage (LoRA)

Trains a LoRA adapter that turns one support ticket into strict triage JSON, evaluates it on the
validation split, pushes the adapter to the Hugging Face Hub and reloads it from the Hub to prove
the upload worked.

**How to run:** see `notebooks/README.md`. In short: add a Colab secret named `HF_TOKEN`
(write access), select the T4 GPU runtime, then *Runtime > Run all*. Nothing else needs editing.

The data is synthetic (see `data/DATA_CARD.md`). The test split is never used here.

## 1. Settings

In [ ]:
# The only cell you may need to edit.
HF_TOKEN_SECRET = "HF_TOKEN"  # name of the Colab secret that holds a Hugging Face write token
ADAPTER_REPO = "Khalid-Mehmood-117/qwen2.5-0.5b-ticket-triage-lora"  # where the adapter is pushed
GITHUB_REPO = "https://github.com/Khalid-Mehmood-117/fine-tuned-llm-ticket-triage.git"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
PUSH_TO_HUB = True

# Training hyperparameters
SEED = 42
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4          # per step; effective batch = BATCH_SIZE * GRAD_ACCUM = 16
GRAD_ACCUM = 4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MAX_LENGTH = 1024       # tokens per example; the data check below shows the longest example

# SMOKE_TEST=1 (environment variable) runs a tiny CPU check without pushing. Never set in Colab.
import os
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
if SMOKE_TEST:
    EPOCHS, PUSH_TO_HUB = 1, False

## 2. Install pinned libraries

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if "peft" in sys.modules or "torchao" in sys.modules:
    raise RuntimeError("Libraries from an earlier run are still loaded. "
                       "Use Runtime > Restart session, then Runtime > Run all.")

PACKAGES = ["transformers==5.18.0", "peft==0.21.2", "trl==1.14.1", "datasets==5.1.0", "accelerate==1.15.0", "huggingface_hub==1.33.0"]
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    # Colab preinstalls torchao 0.10, and peft refuses to import with torchao below 0.16.
    # Training does not use torchao, so remove it before anything imports peft.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)

importlib.invalidate_caches()
if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError("torchao is still installed and breaks peft. Run "
                       "'pip uninstall -y torchao', restart the session, then Run all.")
print("Colab" if IN_COLAB else "Local run", "| packages:", ", ".join(PACKAGES), "| torchao: not installed")

## 3. Check the GPU

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.1f} GB")
elif not SMOKE_TEST:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then Run all again.")
else:
    print("No GPU: smoke test on CPU")
USE_GPU = torch.cuda.is_available()

## 4. Hugging Face login (token from Colab secrets, never printed)

In [ ]:
from huggingface_hub import whoami

if IN_COLAB:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get(HF_TOKEN_SECRET)
    except Exception as error:
        raise RuntimeError(
            f"Could not read the Colab secret {HF_TOKEN_SECRET!r}. Open the key icon in the left "
            "sidebar, add it with a Hugging Face write token and turn on 'Notebook access'.") from error
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if PUSH_TO_HUB:
    print("Logged in to Hugging Face as", whoami(token=HF_TOKEN)["name"])

## 5. Get the code and data

In [ ]:
import json
from pathlib import Path

if IN_COLAB:
    REPO_DIR = Path("/content/fine-tuned-llm-ticket-triage")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_DIR / "src"))

from triage.parse import parse_triage
from triage.prompts import finetuned_messages, target_json
from triage.qwen_local import generate
from triage.rules import apply_rules
from triage.schema import LABEL_FIELDS

COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()


def load_split(name):
    with open(REPO_DIR / "data" / "splits" / f"{name}.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_rows, val_rows = load_split("train"), load_split("validation")
if SMOKE_TEST:
    train_rows, val_rows = train_rows[:16], val_rows[:4]
print(f"repo commit {COMMIT}: {len(train_rows)} train, {len(val_rows)} validation tickets")

## 6. Build the training examples

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def to_example(row):
    """Prompt is system + ticket; completion is the exact JSON answer. Loss is on the completion only."""
    return {"prompt": finetuned_messages(row["ticket"]),
            "completion": [{"role": "assistant", "content": target_json(row["labels"])}]}


train_ds = Dataset.from_list([to_example(r) for r in train_rows])
val_ds = Dataset.from_list([to_example(r) for r in val_rows])

def token_count(example):
    text = tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False)
    return len(tokenizer(text)["input_ids"])


lengths = sorted(token_count(e) for e in train_ds)
print(f"tokens per example: median {lengths[len(lengths) // 2]}, max {lengths[-1]}")
assert lengths[-1] <= MAX_LENGTH, "Raise MAX_LENGTH in the settings cell: some examples would be cut"
print("Example target:", train_ds[0]["completion"][0]["content"])

## 7. Train the LoRA adapter

In [ ]:
import time

from peft import LoraConfig
from transformers import AutoModelForCausalLM, EarlyStoppingCallback, set_seed
from trl import SFTConfig, SFTTrainer

set_seed(SEED)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float32)

lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                         target_modules="all-linear", task_type="CAUSAL_LM")
args = SFTConfig(
    output_dir="outputs",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,               # a float means a fraction of all steps
    gradient_checkpointing=False,    # a 0.5B model fits on a T4 without it, and training is faster
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    fp16=USE_GPU,                    # T4 supports fp16 but not bf16
    bf16=False,
    use_cpu=not USE_GPU,
    seed=SEED,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                     processing_class=tokenizer, peft_config=lora_config,
                     callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
trainer.model.print_trainable_parameters()

if USE_GPU:
    torch.cuda.reset_peak_memory_stats()
start = time.time()
train_result = trainer.train()
TRAIN_MINUTES = (time.time() - start) / 60
PEAK_GPU_GB = torch.cuda.max_memory_allocated() / 1e9 if USE_GPU else 0.0
print(f"training took {TRAIN_MINUTES:.1f} min, peak GPU memory {PEAK_GPU_GB:.1f} GB")

## 8. Training loss and validation metrics

In [ ]:
history = trainer.state.log_history
print("Training loss")
for entry in history:
    if "loss" in entry:
        print(f"  step {entry['step']:>4}  epoch {entry['epoch']:.2f}  loss {entry['loss']:.4f}")
print("Validation loss")
for entry in history:
    if "eval_loss" in entry:
        print(f"  epoch {entry['epoch']:.2f}  eval_loss {entry['eval_loss']:.4f}")
BEST_EVAL_LOSS = trainer.state.best_metric

# Generate on the validation split with greedy decoding and score every field.
model = trainer.model
model.eval()
start = time.time()
raw_outputs = generate(model, tokenizer, [r["ticket"] for r in val_rows], batch_size=16)
seconds_per_ticket = (time.time() - start) / len(val_rows)

parsed = [parse_triage(text) for text in raw_outputs]
valid = [p for p, _ in parsed if p is not None]
VAL_METRICS = {"tickets": len(val_rows), "json_valid": round(len(valid) / len(val_rows), 3)}
for field in LABEL_FIELDS:
    correct = sum(1 for (p, _), row in zip(parsed, val_rows) if p and p[field] == row["labels"][field])
    VAL_METRICS[f"{field}_accuracy"] = round(correct / len(val_rows), 3)
all_correct = sum(1 for (p, _), row in zip(parsed, val_rows)
                  if p and all(p[f] == row["labels"][f] for f in LABEL_FIELDS))
VAL_METRICS["all_fields_accuracy"] = round(all_correct / len(val_rows), 3)
VAL_METRICS["seconds_per_ticket"] = round(seconds_per_ticket, 3)

print("\nValidation metrics (raw model output, before business rules)")
for key, value in VAL_METRICS.items():
    print(f"  {key:28} {value}")
invalid = [(row["id"], err, text) for (p, err), row, text in zip(parsed, val_rows, raw_outputs) if p is None]
for row_id, err, text in invalid[:3]:
    print(f"\ninvalid output {row_id}: {err}\n{text[:300]}")

## 9. Push the adapter to the Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

ADAPTER_DIR = Path("adapter")
model.save_pretrained(ADAPTER_DIR)
summary = {
    "base_model": BASE_MODEL,
    "repo_commit": COMMIT,
    "train_examples": len(train_rows),
    "validation_examples": len(val_rows),
    "epochs_run": round(trainer.state.epoch, 2),
    "best_eval_loss": BEST_EVAL_LOSS,
    "train_minutes": round(TRAIN_MINUTES, 1),
    "peak_gpu_gb": round(PEAK_GPU_GB, 2),
    "gpu": torch.cuda.get_device_name(0) if USE_GPU else "cpu",
    "hyperparameters": {"lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
                        "learning_rate": LEARNING_RATE, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
                        "grad_accum": GRAD_ACCUM, "max_length": MAX_LENGTH, "seed": SEED},
    "validation_metrics": VAL_METRICS,
}
(ADAPTER_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

metrics_table = "\n".join(f"| {k} | {v} |" for k, v in VAL_METRICS.items())
(ADAPTER_DIR / "README.md").write_text(f"""---
base_model: {BASE_MODEL}
library_name: peft
license: apache-2.0
pipeline_tag: text-generation
tags: [lora, peft, trl, customer-support, ticket-triage, json]
---

# Qwen2.5-0.5B ticket triage LoRA adapter

LoRA adapter for [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}) that reads one customer
support ticket and returns strict JSON: category, priority, sentiment, needs_human,
suggested_action and a short reason. Trained on synthetic tickets generated with gpt-4o-mini.
Code, data card and evaluation: {GITHUB_REPO.removesuffix(".git")}

Business rules (safety, legal, security, chargeback) are applied in code after the model and
are not part of this adapter. Use the system prompt from `src/triage/prompts.py`.

## Training
TRL SFTTrainer, LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}, all linear layers,
lr {LEARNING_RATE}, cosine schedule, effective batch {BATCH_SIZE * GRAD_ACCUM}, up to {EPOCHS}
epochs with early stopping on validation loss, fp16 on {summary["gpu"]}, loss on the answer only.
{len(train_rows)} training and {len(val_rows)} validation tickets, repo commit {COMMIT}.
Best validation loss {BEST_EVAL_LOSS:.4f}. Training time {TRAIN_MINUTES:.1f} min.

## Validation metrics (greedy decoding, before business rules)
| metric | value |
|---|---|
{metrics_table}

Test set results are in the GitHub repo (eval/results.md).
""", encoding="utf-8")

if PUSH_TO_HUB:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(ADAPTER_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=str(ADAPTER_DIR), repo_id=ADAPTER_REPO,
                      commit_message=f"Adapter trained from repo commit {COMMIT}")
    print(f"Pushed to https://huggingface.co/{ADAPTER_REPO}")
else:
    print(f"Smoke test: adapter saved to {ADAPTER_DIR.resolve()}, not pushed")

## 10. Reload the adapter from the Hub and run 3 sample tickets

In [ ]:
import gc

from triage.qwen_local import load_model

del model, trainer
gc.collect()
if USE_GPU:
    torch.cuda.empty_cache()

source = ADAPTER_REPO if PUSH_TO_HUB else str(ADAPTER_DIR)
hub_model, hub_tokenizer = load_model(adapter=source, base_model=BASE_MODEL, token=HF_TOKEN)
print(f"Loaded adapter from {source}\n")

SAMPLES = [
    "My Glowhome space heater started sparking and smoking last night. I unplugged it but my hand got burned.",
    "Hallo, ich wurde diesen Monat zweimal für mein Abo belastet. Bitte erstatten Sie mir eine Abbuchung.",
    "where is my order??",
]
for ticket, text in zip(SAMPLES, generate(hub_model, hub_tokenizer, SAMPLES)):
    triage, error = parse_triage(text)
    print("TICKET:", ticket)
    print("MODEL :", text)
    if triage:
        final, hits = apply_rules(ticket, triage)
        print("VALID : yes | rules fired:", [h.rule for h in hits] or "none")
    else:
        print("VALID : no |", error)
    print()

## 11. Summary to send back

In [ ]:
print(json.dumps({k: summary[k] for k in ["repo_commit", "gpu", "train_minutes", "peak_gpu_gb",
                                          "epochs_run", "best_eval_loss", "validation_metrics"]},
                 indent=2))
print(f"\nAdapter: https://huggingface.co/{ADAPTER_REPO}" if PUSH_TO_HUB else "\nSmoke test finished")